# Détection des maladies de la feuille de tomate — Entraînement (notebook)

**Projet universitaire de Deep Learning** — MobileNetV2, Transfer Learning, Flask.

Ce notebook résume les étapes d'entraînement et pointe vers les scripts du dossier
`training/`. Il ne remplace pas les scripts, il les accompagne pour la lecture.

Prérequis : `pip install -r requirements.txt` et le dataset préparé dans
`data/processed/`.

### Étapes

1. Préparation du dataset — `python training/prepare_data.py`
2. Entraînement — `python training/train.py`
3. Évaluation — `python training/evaluate.py`
4. Inférence temps réel — `python app.py` (Flask)

## 1. Vérifier que le dataset est prêt

Le dataset doit déjà avoir été préparé (train/validation/test dans `data/processed/`).

In [ ]:
import os, sys
from pathlib import Path

# Permet d'importer depuis la racine du projet
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import config

print("Classes :", config.CLASS_NAMES)
print("Image size :", config.IMAGE_SIZE)

for split in ["train", "validation", "test"]:
    d = config.PROCESSED_DATA_DIR / split
    exists = d.exists()
    if exists:
        n = sum(1 for p in d.iterdir() if p.is_dir())
        print(f"{split:>12} : {n} classes")
    else:
        print(f"{split:>12} : absent (à préparer avec training/prepare_data.py)")

## 2. Entraînement

L'entraînement complet est fait par `training/train.py` (deux phases : extraction de
caractéristiques puis fine-tuning).

**Sur CPU (version rapide)** :

```bash
python training/train.py --epochs 15 --fine-tune-epochs 8
```

Les résultats officiels du livrable viennent de cette commande. Les métriques et figures
sont produites ensuite par `training/evaluate.py`.

## 3. Vérifier les résultats (s'ils existent)

Si `training/evaluate.py` a déjà été exécuté, on peut afficher les métriques.

In [ ]:
from utils.helpers import charger_json
from pathlib import Path

eval_path = config.METRICS_DIR / "evaluation.json"

if eval_path.exists():
    import pandas as pd
    d = charger_json(eval_path)
    print("Accuracy :", d["global"]["accuracy"])
    print("F1 macro :", d["global"]["f1_macro"])
    print("Nb erreurs :", d["global"]["nb_erreurs"], "/", d["global"]["nb_images"])
else:
    print("Aucune métrique : exécuter d'abord training/evaluate.py")

## 4. Inférence sur une image (test rapide)

Permet de voir ce que retourne le modèle sur une image de test.

In [ ]:
from utils.predictor import obtenir_predictor
from utils.helpers import nom_affichage

predictor = obtenir_predictor()
predictor.charger()

# Prendre la première image de test
test_dir = config.PROCESSED_DATA_DIR / "test"
first_class = config.CLASS_NAMES[0]
img_path = next(test_dir.glob("**/*.jpg"), None) or next(test_dir.glob("**/*.png"), None)

if img_path:
    resultat = predictor.predire(img_path)
    print("Classe prédite :", nom_affichage(resultat["classe"]))
    print("Confiance :", resultat["confiance"])
    print("Durée :", resultat["duree_ms"], "ms")
else:
    print("Aucune image de test trouvée.")

## 5. L'application Web (inférence temps réel)

Une fois le modèle entraîné, l'application Flask se lance avec :

```bash
python app.py
```

Elle s'ouvre sur http://127.0.0.1:5000. L'interface reste volontairement minimale :
téléversement → prédiction → résultat (sans base de données, sans historique).